# Hybrid SZ3 + Quantization Experiments

Tests the **HybridCompressor**: SZ3 (eb=0.001) for multi-dimensional tensors
(convolutional weights), 8-bit quantization for 1-D tensors (BatchNorm params, biases).

ResNet-20 breakdown:
- 19 multi-dimensional tensors (~98% of all parameters) → SZ3
- 46 one-dimensional tensors (~2% of parameters) → 8-bit quantization

Runs: `hybrid_sz0.001_q8bit` seed=0 × 200 rounds on CIFAR-10.

**First run:** Cell 1 → restart runtime → Cell 2 onward.

**Resuming after timeout:** Mount Drive (Cell 2) → re-run Cell 4.

In [ ]:
%%bash
# Cell 1: install dependencies and patch flwr numpy compatibility
pip install -q 'flwr[simulation]==1.9.0' ray pandas
pip install -q 'protobuf>=4.23' --force-reinstall
pip install -q pysz
# flwr 1.9.0 uses np.float_ which was removed in numpy 2.0
find /usr/local/lib /usr/lib -path '*/flwr/common/typing.py' 2>/dev/null \
    -exec sed -i 's/np\.float_/np.float64/g' {} \; -print
echo ''
echo 'Done. Now: Runtime -> Restart runtime, then run Cell 2.'

In [ ]:
# Cell 2: mount Drive and clone/pull repo
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess

REPO_URL  = 'https://github.com/ayananyways/fl-compression-study.git'
CLONE_DIR = '/content/drive/MyDrive/fl-compression-study'

if not os.path.exists(CLONE_DIR):
    subprocess.run(['git', 'clone', REPO_URL, CLONE_DIR], check=True)
    print('Cloned.')
else:
    subprocess.run(['git', '-C', CLONE_DIR, 'pull'], check=True)
    print('Pulled latest.')

os.chdir(CLONE_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Cell 3: pre-flight checks
import sys, os

ok = True

try:
    import pysz
    print('pysz OK')
except ImportError:
    print('ERROR: pysz not found — run Cell 1 and restart runtime')
    ok = False

for path in [
    'fl-flower/run.py',
    'src/compressors/hybrid.py',
    'src/compressors/sz.py',
    'src/compressors/quantization.py',
]:
    if os.path.exists(path):
        print(f'{path} OK')
    else:
        print(f'ERROR: {path} missing — commit and push the latest code first')
        ok = False

with open('fl-flower/run.py') as f:
    run_src = f.read()
if '"hybrid"' in run_src:
    print('run.py has hybrid compressor choice OK')
else:
    print('ERROR: run.py does not have hybrid choice — push latest run.py')
    ok = False

sys.path.insert(0, '.')
try:
    from src.compressors.hybrid import HybridCompressor
    import torch
    c = HybridCompressor(sz_error_bound=0.001, quant_bits=8)
    # large 2D tensor → SZ path
    t2d = torch.randn(64, 16, 3, 3)
    r2d = c.decompress(c.compress(t2d), t2d.shape, t2d.dtype)
    assert r2d.shape == t2d.shape
    assert (t2d - r2d).abs().max().item() <= 0.001 + 1e-6, 'SZ error bound violated'
    # 1D tensor → quant path
    t1d = torch.randn(64)
    r1d = c.decompress(c.compress(t1d), t1d.shape, t1d.dtype)
    assert r1d.shape == t1d.shape
    print('HybridCompressor smoke test passed')
except Exception as e:
    print(f'ERROR in HybridCompressor: {e}')
    ok = False

if ok:
    print('\nAll checks passed — ready to run.')
else:
    print('\nFix errors above before running Cell 4.')

In [ ]:
# Cell 4: run hybrid experiment (seed=0, 200 rounds)
# Each round prints immediately as it completes:
#   [server] round  1  acc=15.43%  loss=2.2984  ratio=3.44x
# CSV on Drive is also updated after each round.

import subprocess, sys, os

OUTPUT = 'results/resnet20_cifar10_hybrid.csv'
os.makedirs('results', exist_ok=True)

label = 'hybrid_sz0.001_q8bit  seed=0'
print(f'\n{"="*60}')
print(f'  {label}')
print(f'{"="*60}', flush=True)

cmd = [
    sys.executable, 'fl-flower/run.py',
    '--dataset',      'cifar10',
    '--compressor',   'hybrid',
    '--error-bound',  '0.001',
    '--bits',         '8',
    '--seed',         '0',
    '--rounds',       '200',
    '--num-clients',  '10',
    '--local-epochs', '2',
    '--output',       OUTPUT,
]

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if proc.returncode != 0:
    print(f'\nFAILED (exit {proc.returncode})')
else:
    print(f'\nDone: {label}')

In [ ]:
# Cell 5: check results and preview accuracy
import pandas as pd, os

OUTPUT = 'results/resnet20_cifar10_hybrid.csv'

if not os.path.exists(OUTPUT):
    print('Output file not found.')
else:
    df = pd.read_csv(OUTPUT, on_bad_lines='skip')
    df['round'] = pd.to_numeric(df['round'], errors='coerce')
    df['val_accuracy'] = pd.to_numeric(df['val_accuracy'], errors='coerce')
    df['seed'] = pd.to_numeric(df['seed'], errors='coerce')

    print(f'Output file: {OUTPUT}  ({len(df)} rows)\n')
    for comp in sorted(df['compressor'].unique()):
        sub = df[df['compressor'] == comp]
        for s in sorted(sub['seed'].dropna().unique()):
            rows = sub[sub['seed'] == s]
            max_r = int(rows['round'].max())
            peak  = rows['val_accuracy'].max()
            ratio = pd.to_numeric(rows['compression_ratio'], errors='coerce').mean()
            status = 'COMPLETE' if max_r >= 200 else f'PARTIAL ({max_r}/200)'
            print(f'  {comp} s{int(s)}: {status}, peak={peak:.1f}%, '
                  f'avg_ratio={ratio:.2f}x')